# Предрасчёт данных коллизий

По образцу `TreeCreator.ipynb`.

- **object1** — объект-отсчёт в `(0, 0)`, скорость `(0, 0)`.
- **object2** — вторая форма, может быть повёрнута.
- В txt только случаи с коллизией `x y dx dy`.
- В C# по этим строкам строится дерево: дошли до листа — коллизия есть.

In [ ]:
import math

ship_grid = [[0, 1, 0], [1, 1, 1], [1, 0, 1]]
torpedo_grid = [[0, 1, 0], [0, 1, 0], [0, 0, 0]]

name_obj1 = "ship"
name_obj2 = "torpedo"

In [ ]:
def place_object_on_grid(grid, obj, x, y):
    obj_height = len(obj)
    obj_width = len(obj[0])
    for i in range(obj_height):
        for j in range(obj_width):
            grid[x + i][y + j] += obj[i][j]
    return grid


def check_collision(object1, object2, grid_size_x, grid_size_y, pos_x, pos_y):
    width = 4 * grid_size_x + len(object2[0])
    height = 4 * grid_size_y + len(object2)
    grid = [[0 for _ in range(width)] for _ in range(height)]

    origin = 2 * grid_size_x
    grid = place_object_on_grid(grid, object1, origin, origin)
    grid = place_object_on_grid(grid, object2, origin - pos_x, origin + pos_y)

    return any(cell == 2 for row in grid for cell in row)

In [ ]:
def brezenham_algorithm(point1_x, point2_x, point1_y, point2_y):
    points = [(point1_x, point1_y), (point2_x, point2_y)]
    deltax = abs(point2_x - point1_x)
    deltay = abs(point2_y - point1_y)
    error = 0
    deltaerr = deltay + 1
    y = point1_y
    diry = point2_y - point1_y
    if diry > 0:
        diry = 1
    elif diry < 0:
        diry = -1
    for x in range(point1_x, point2_x + 1):
        points.append((x, y))
        error += deltaerr
        if error >= deltax + 1:
            y = y + diry
            error -= deltax + 1
    return points


def turning(object_grid, angle):
    size_x = len(object_grid[0])
    size_y = len(object_grid)
    points = []
    for column in range(size_x):
        start = -1
        end = -1
        for row in range(size_y):
            if object_grid[row][column] == 1:
                if start == -1:
                    start = row
                end = row
        if start != -1 and end != -1:
            center_x = size_x // 2
            center_y = size_y // 2
            new_start_x = (column - center_x) * math.cos(math.radians(angle)) - (start - center_y) * math.sin(math.radians(angle))
            new_start_y = (column - center_x) * math.sin(math.radians(angle)) + (start - center_y) * math.cos(math.radians(angle))
            new_end_x = (column - center_x) * math.cos(math.radians(angle)) - (end - center_y) * math.sin(math.radians(angle))
            new_end_y = (column - center_x) * math.sin(math.radians(angle)) + (end - center_y) * math.cos(math.radians(angle))
            new_start_x += center_x
            new_start_y += center_y
            new_end_x += center_x
            new_end_y += center_y
            points += brezenham_algorithm(int(round(new_start_x)), int(round(new_end_x)), int(round(new_start_y)), int(round(new_end_y)))
    return points


def implementing(size_x, size_y, points):
    grid = [[0 for _ in range(size_x)] for _ in range(size_y)]
    for x, y in points:
        grid[y][x] = 1
    return grid

In [ ]:
def making_situations(grid_size_x, grid_size_y, object1, object2, angle_list):
    situations = []
    for i in range(-2 * grid_size_x, 2 * grid_size_x):
        for j in range(-2 * grid_size_y, 2 * grid_size_y):
            for angle in angle_list:
                rotated = implementing(len(object2[0]), len(object2), turning(object2, angle))
                situations.append((i, j, angle, check_collision(object1, rotated, grid_size_x, grid_size_y, i, j)))
    return situations


def making_collision_situations(grid_size_x, grid_size_y, max_speed_x1, max_speed_y1, max_speed_x2, max_speed_y2, object1, object2, angle_list):
    x_y_dx_dy = []
    situations = making_situations(grid_size_x, grid_size_y, object1, object2, angle_list)
    for x in range(-2 * grid_size_x, 2 * grid_size_x):
        for y in range(-2 * grid_size_y, 2 * grid_size_y):
            for i in range(-(max_speed_x1 + max_speed_x2), max_speed_x1 + max_speed_x2):
                for j in range(-(max_speed_y1 + max_speed_y2), max_speed_y1 + max_speed_y2):
                    nearest_angle = min(angle_list, key=lambda a: abs(a - math.atan2(j, i)))
                    if (x, y, nearest_angle, True) in situations:
                        x_y_dx_dy.append((x, y, i, j))
                    elif (x + i, y + j, nearest_angle, True) in situations:
                        x_y_dx_dy.append((x, y, i, j))
    return x_y_dx_dy

In [ ]:
def write_collision_file(path, entries):
    with open(path, "w", encoding="utf-8") as file:
        for x, y, dx, dy in entries:
            file.write(f"{x} {y} {dx} {dy}\n")

In [ ]:
grid_size = 10
max_speed_1 = 5
max_speed_2 = 8
angle_list = [0, 45, 90, 135, 180, 225, 270, 315]

point_grid = [[1]]

ship_and_torpedo = making_collision_situations(
    grid_size, grid_size,
    max_speed_1, max_speed_1, max_speed_2, max_speed_2,
    ship_grid, torpedo_grid, angle_list,
)

point_and_torpedo = making_collision_situations(
    grid_size, grid_size,
    0, 0, max_speed_2, max_speed_2,
    point_grid, torpedo_grid, angle_list,
)

point_and_ship = making_collision_situations(
    grid_size, grid_size,
    0, 0, max_speed_2, max_speed_2,
    point_grid, ship_grid, angle_list,
)

write_collision_file(f"data/{name_obj1}and{name_obj2}.txt", ship_and_torpedo)
write_collision_file("data/torpedo.txt", point_and_torpedo)
write_collision_file("data/ship.txt", point_and_ship)

print(f"{name_obj1}and{name_obj2}: {len(ship_and_torpedo)} collisions")
print(f"torpedo profile: {len(point_and_torpedo)} collisions")
print(f"ship profile: {len(point_and_ship)} collisions")

## Формат txt

Только строки с коллизией:

```
x y dx dy
x y dx dy
```

Каждая строка добавляет ключ в дерево `Dict[x][y][dx] → HashSet<dy>`.
Проверка в C#: `Collides(x, y, dx, dy)`.